In this notebook, we focus on predicting the match result. 

In [1]:
import pandas as pd
import numpy as np

## Preparation

In [2]:
data = pd.read_csv('/Users/tliu/Desktop/Erdos Project/3_Player_Data_Generation/match_data_20_tourns_modified.csv')
data.head()

,player1,player2,best_of,player1_elo,player2_elo,elo_match_win_rate,elo_frame_win_rate,p1_matches_played,p1_matches_won,p1_frames_played,...,p1_frames_played_3_years,p1_frames_won_3_years,p2_frames_played_1_year,p2_frames_won_1_year,p2_frames_played_3_years,p2_frames_won_3_years,score1,score2,match_result,win_percentage
0,Long Zehuang,Haydon Pinhey,7,1328,1158,0.719210,0.604679,78,39,443,...,284,147,121,57,347,168,4,3,0.0,0.571429
1,Wang Yuchen,Andrew Pagett,7,1162,1162,0.500000,0.500000,100,37,626,...,158,89,144,65,417,170,4,1,0.0,0.800000
2,Ben Mertens,Daniel Womersley,7,1262,1109,0.699444,0.594476,107,50,642,...,471,240,86,40,180,89,1,4,1.0,0.200000
3,Paul Deaville,Jimmy White,7,1096,1141,0.438735,0.471905,32,17,158,...,126,66,85,31,384,153,3,4,1.0,0.428571
4,Alexander Ursenbacher,Mostafa Dorgham,7,1302,1031,0.821305,0.663180,286,129,1728,...,400,196,108,41,154,53,4,1,0.0,0.800000


In [3]:
#Add more features
data['p1_frames_win_rate'] = data['p1_frames_won']/ data['p1_frames_played']
data['p2_frames_win_rate'] = data['p2_frames_won']/ data['p2_frames_played']

data['p1_matches_win_rate'] = data['p1_matches_won']/ data['p1_matches_played']
data['p2_matches_win_rate'] = data['p2_matches_won']/ data['p2_matches_played']

#p1_matches_win_rate and p2_matches_win_rate both have missing values
#We will fill them with 0.5
data.fillna(0.5, inplace = True)

#Drop players' names from the data since we won't consider strings as our features for modeling
data = data.drop(['player1', 'player2'], axis = 1)

In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1914 entries, 0 to 1913
Data columns (total 29 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   best_of                   1914 non-null   int64  
 1   player1_elo               1914 non-null   int64  
 2   player2_elo               1914 non-null   int64  
 3   elo_match_win_rate        1914 non-null   float64
 4   elo_frame_win_rate        1914 non-null   float64
 5   p1_matches_played         1914 non-null   int64  
 6   p1_matches_won            1914 non-null   int64  
 7   p1_frames_played          1914 non-null   int64  
 8   p1_frames_won             1914 non-null   int64  
 9   p2_matches_played         1914 non-null   int64  
 10  p2_matches_won            1914 non-null   int64  
 11  p2_frames_played          1914 non-null   int64  
 12  p2_frames_won             1914 non-null   int64  
 13  p1_frames_played_1_year   1914 non-null   int64  
 14  p1_frame

In [5]:
data.columns

Index(['best_of', 'player1_elo', 'player2_elo', 'elo_match_win_rate',
       'elo_frame_win_rate', 'p1_matches_played', 'p1_matches_won',
       'p1_frames_played', 'p1_frames_won', 'p2_matches_played',
       'p2_matches_won', 'p2_frames_played', 'p2_frames_won',
       'p1_frames_played_1_year', 'p1_frames_won_1_year',
       'p1_frames_played_3_years', 'p1_frames_won_3_years',
       'p2_frames_played_1_year', 'p2_frames_won_1_year',
       'p2_frames_played_3_years', 'p2_frames_won_3_years', 'score1', 'score2',
       'match_result', 'win_percentage', 'p1_frames_win_rate',
       'p2_frames_win_rate', 'p1_matches_win_rate', 'p2_matches_win_rate'],
      dtype='object')

In [6]:
#Train test split
from sklearn.model_selection import train_test_split
data_train, data_test = train_test_split(data, 
                                        test_size = 0.2,
                                        shuffle = True,
                                        random_state=216)

# data_train, data_cv = train_test_split(data_train, 
#                                         test_size = 0.2,
#                                         shuffle = True,
#                                         random_state=216)

# #Get the cv indices
# cv = data_cv.index.values

In [7]:
#Create predictors and targets for training and cross-validation set
y_train = data_train['match_result']
# y_cv = data_cv['match_result']
y_test = data_test['match_result']

#To get the predictor, we exclude players' names (Strings) and match results.
X_train = data_train.drop(['match_result', 'win_percentage', 'score1', 'score2'], axis = 1)
# X_cv = data_cv.drop(['match_result', 'win_percentage','score1', 'score2'], axis = 1)
X_test = data_test.drop(['match_result', 'win_percentage','score1', 'score2'], axis = 1)

In [8]:
# #Create predictors and targets for training and test set
# result_train = data_train['match_result']
# win_perc_train = data_train['win_percentage']
# #exclude players' names and match results.
# X_train = data_train.drop(['match_result', 'win_percentage', 'player1', 'player2', 'score1', 'score2'], axis = 1)


# result_test = data_test['match_result']
# win_perc_test = data_test['win_percentage']
# X_test = data_test.drop(['match_result', 'win_percentage', 'player1', 'player2', 'score1', 'score2'], axis = 1)

## Performance Metrics
Because the two classes in the target are symmetric (swapping player1 and player2 will exchange positive and negative but still represents the same match), we won't consider metrics such as presision, specificity and sensitivity since they are the same as accuracy score. We will only consider accuracy score.

In [9]:
#Import metrics
from sklearn.metrics import accuracy_score
from sklearn.model_selection import KFold

In [10]:
#Create dictionaries to store the metrics.
accuracy_scores = {}
avg_scores = {}

#Create a list to store all the models we consider.
models = {}

In [11]:
def print_avg_cv_metrics(model, model_name):
    """
    Given a model, computes and stores accuracy scores and presision across
    the CV splits. We will do 5 splits using KFold from sklearn.
    """

    #Create an empty array to store the scores.
    print('Currently working on ' + model_name + '.')

    scores = np.zeros(5)

    kfold = KFold(n_splits = 5, random_state = 216, shuffle = True)

    #Use KFold to further splits the training data into training set and cross-validation set.
    for i, (train_index, cv_index) in enumerate(kfold.split(data_train, y_train)):
        #Get the  raining set and cross-validation set
        df_tt = data_train.iloc[train_index]
        df_ho = data_train.iloc[cv_index]

        #Create predictors and targets for training and cross-validation set
        y_tt = df_tt['match_result']
        y_ho = df_ho['match_result']

        #To get the predictor, we exclude players' names (Strings) and match results.
        X_tt = df_tt.drop(['match_result', 'win_percentage', 'score1', 'score2'], axis = 1)
        X_ho = df_ho.drop(['match_result', 'win_percentage', 'score1', 'score2'], axis = 1)
        

        #Fit and Predict
        model.fit(X_tt, y_tt)
        prediction = model.predict(X_ho)

        acc_score = accuracy_score(y_ho, prediction)
        scores[i] = acc_score

    print('The scores for model ' + model_name + ' are:')
    print(scores)

    avg_score = np.mean(scores)
    print(f'The average score is: {avg_score}.')

    accuracy_scores[model_name] = scores
    avg_scores[model_name] = avg_score

    models[model_name] = model




In [12]:
def print_metrics(models):
    """
    Print the accuracy scores and presion for all the model tested.
    """
    return None


## Model 0: Prediction by elo rating

## Model 1: Logistic Regression with PCA

In [13]:
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler

In [14]:
#Get the number of features in X_train
print(len(X_train.columns))

25


In [15]:
scaler = StandardScaler()
pca = PCA()
log_reg = LogisticRegression(max_iter=10000)

log_reg_with_pca = Pipeline([('scale', scaler),
                    ('pca', pca),
                    ('logistic', log_reg)])

param_grid = {
    "pca__n_components": list(range(5, 26, 5)),
    "logistic__C": np.logspace(-4, 4, 8),
}


grid_search = GridSearchCV(log_reg_with_pca,
                           param_grid=param_grid,
                           scoring='accuracy',
                           cv=5)
grid_search.fit(X_train, y_train.values)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('scale', StandardScaler()),
                                       ('pca', PCA()),
                                       ('logistic',
                                        LogisticRegression(max_iter=10000))]),
             param_grid={'logistic__C': array([1.00000000e-04, 1.38949549e-03, 1.93069773e-02, 2.68269580e-01,
       3.72759372e+00, 5.17947468e+01, 7.19685673e+02, 1.00000000e+04]),
                         'pca__n_components': [5, 10, 15, 20, 25]},
             scoring='accuracy')

Notice that for linear models, pca__n_components = number_of_features is the same as modeling without PCA. 

In [16]:
print(grid_search.best_params_)
print(grid_search.best_score_)

{'logistic__C': np.float64(0.2682695795279725), 'pca__n_components': 10}
0.6557663239019821


In [17]:
print_avg_cv_metrics(grid_search, 'Logistic Regression with PCA')

Currently working on Logistic Regression with PCA.
The scores for model Logistic Regression with PCA are:
[0.62214984 0.66666667 0.67647059 0.6372549  0.63398693]
The average score is: 0.6473057844201742.


## Model 2: Random Forest

In [18]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
rf = RandomForestClassifier()

param_grid = {
    "max_depth": [3, 4 ,5 ,6 ,7 ,8],
    "n_estimators": np.linspace(100, 900, 9).astype(int)
}


grid_search = GridSearchCV(random_forest_with_pca,
                           param_grid=param_grid,
                           scoring='accuracy',
                           cv=5)
grid_search.fit(X_train, y_train)

KeyboardInterrupt: 